# TODO 6.4：INT8 质量、长上下文和结构化输出

本 Notebook 只对真实 INT8 导出模型评测。没有模型 manifest 时会明确输出 `not_run`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import gc, json, sys
from pathlib import Path
PROJECT_DIR=Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR=Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST=Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
INT8_DIR=Path('/content/drive/MyDrive/banking_llm_project/int8_fp8')
RESULT_DIR=PROJECT_DIR/'out/int8_fp8'
BASELINE_CONFIG=PROJECT_DIR/'configs/baseline.json'
QUALITY_CONFIG=PROJECT_DIR/'configs/baseline_quality.json'
sys.path.insert(0,str(PROJECT_DIR))
from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.quantization.int8_fp8 import check_structured_json, evaluate_int8_outputs
assets=load_assets(DATASET_DIR)
config=json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))

In [ ]:
manifest=INT8_DIR/'smoothquant-int8-w8a8/quantized_model_manifest.json'
if not manifest.is_file():
    payload={'status':'not_run','detail':'No real SmoothQuant INT8 manifest found. Run int8_fp8_experiment_colab.ipynb first.'}
    (RESULT_DIR/'evaluation.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print(json.dumps(payload,ensure_ascii=False,indent=2))
    raise RuntimeError(payload['detail'])
print('INT8 manifest:',manifest)

In [ ]:
service=BaselineService.from_local_model(BASELINE_CONFIG,manifest,RESULT_DIR/'int8_quality_service.jsonl')
predictions=[]; references=[]
for start in range(0,len(assets.quality_dev),int(config['batch_size'])):
    rows=[assets.quality_dev[index] for index in range(start,min(start+int(config['batch_size']),len(assets.quality_dev)))]
    requests=[InferenceRequest.from_finqa_row(row,request_id=f'int8-quality-{start+offset+1:06d}') for offset,row in enumerate(rows)]
    results=service.generate_batch(requests,prompt_suffix=config.get('prompt_suffix',''))
    predictions.extend(result['output_text'] for result in results)
    references.extend(str(row.get('answer') or row.get('qa',{}).get('answer')) for row in rows)
long_results=[]
for token_limit in (128,512,2048):
    service.config['max_input_tokens']=token_limit
    result=service.generate_one(InferenceRequest.from_finqa_row(assets.quality_dev[0],request_id=f'int8-long-{token_limit}'))
    long_results.append({'input_tokens':token_limit,'passed':bool(result.get('output_text','').strip()),'output_length':len(result.get('output_text',''))})
structured_results=[]
for index,row in enumerate(assets.quality_dev.select(range(3))):
    result=service.generate_one(InferenceRequest.from_finqa_row(row,request_id=f'int8-structured-{index+1}'),prompt_suffix='\nReturn JSON with fields answer and unit only.')
    structured_results.append({'index':index,**check_structured_json(result['output_text'],['answer','unit'])})
payload=evaluate_int8_outputs(predictions,references,long_results,structured_results,float(config.get('tolerance',1e-4)))
payload['status']='completed'
(RESULT_DIR/'evaluation.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(payload,ensure_ascii=False,indent=2))